# Data exploration

What is in the four raw files, what is wrong with them, and how they fit together.
The fixes themselves live in `src/`. This notebook shows the evidence behind each one,
and the decisions are recorded in `ASSUMPTIONS.md`.

In [1]:
import math
import sys
from pathlib import Path

# the notebook runs from notebooks/, so add the repo root to find src/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import DATA_DIR, FORECAST_MONTHS, GRAIN, PDCS, RAW_FILES
from src.load_raw import read_all_raw
from src.standardize import standardize_all

pd.set_option("display.max_columns", 30)

## 1. First look at each file

Every column is read as text, exactly as received (`src/load_raw.py`), so nothing is converted yet.

In [2]:
raw = read_all_raw()
for name, table in raw.items():
    print(f"{name}: {table.shape[0]} rows, {table.shape[1]} columns")

parts: 101 rows, 5 columns
inventory: 309 rows, 4 columns
supply: 113 rows, 7 columns
demand: 516 rows, 28 columns


**parts.csv**: the part master. One row per part. `qpc` is the number of pieces on one standard pallet.

In [3]:
raw["parts"].head()

,part_number,part_description,part_type,planner,qpc
0,SKU2,Bolt M8 LH,Fastener,Alex Rivera,250
1,SKU72,Strut Assy,Suspension,Sofia Novak,20
2,SKU56,Windshield Gen2,Glass,Daniel Okafor,10
3,SKU6,Fuse Box,Electrical,Sofia Novak,50
4,SKU57,Compressor,HVAC,Priya Patel,


**inventory_on_hand.csv**: pieces on hand on 2026-10-22. One row per part and location.

In [4]:
raw["inventory"].head()

,part_number,location,on_hand_qty,snapshot_date
0,SKU26,PDC3,312,2026-10-22
1,SKU4,PDC-3,38.0,2026-10-22
2,SKU55,GDC,207,2026-10-22
3,SKU37,PDC3,1,2026-10-22
4,SKU65,PDC1,4,2026-10-22


**incoming_supply.csv**: open supply. One row per PO line. `source` is a supplier PO,
or a GDC Transfer that has already left the GDC and is in transit to a PDC.

In [5]:
raw["supply"].head()

,po_number,po_line,part_number,destination,source,qty,eta_date
0,PO1036,10,SKU0022,PDC2,Supplier,89,11/15/2026
1,PO1009,10,SKU-53,PDC1,Supplier,24,2026-11-13
2,PO1001,10,SKU82,PDC2,GDC Transfer,7,2026-12-10
3,PO1051,20,SKU6,PDC1,GDC Transfer,37,2026-12-18
4,PO1011,10,SKU43,PDC3,GDC Transfer,4432,17-Nov-2026


**demand_forecast_monthly.csv**: 24 months of forecast, one column per month.
One row per part, location and demand type. `Direct` is customer demand at that location.
`Replenishment` rows sit at the GDC and are its planned shipments to the PDC named in `ship_to`.

In [6]:
raw["demand"].head()

,part_number,location,demand_type,ship_to,2026-11,2026-12,2027-01,2027-02,2027-03,2027-04,2027-05,2027-06,2027-07,2027-08,2027-09,2027-10,2027-11,2027-12,2028-01,2028-02,2028-03,2028-04,2028-05,2028-06,2028-07,2028-08,2028-09,2028-10
0,SKU84,PDC2,Direct,,26,28,26,36,38,45,32,40,39,31,30,25,30,24,24,21,46,43,42,41,36,42,30,30
1,SKU89,GDC,Direct,,150,204,208,196,216,234,188.0,162.0,190,153,144,176,163,173,171,195,207,180,197,161,178,138.0,141,173
2,SKU68,GDC,Replenishment,PDC3,40,0,40,0,0,40.0,0,0,0,40,0,0,0,40,0,0,0,40,0,0,0,0,40.0,0
3,SKU24,PDC 2,Direct,,17,21,12.0,14,12,9,8,14,19,16.0,13,22,23,17.0,14,18,21.0,19.0,15,22.0,21,18,26,14
4,SKU94,GDC,Direct,,106.0,116,103,85,114,118,109,97,86,102,90,113,110,98,115,137,116,136.0,92,88,82,88,84,111


**What we see:** even the first five rows show the mess: part numbers like `SKU0022` and `SKU-53`,
locations like `PDC-3` and `PDC 2`, quantities like `38.0`, three date layouts, and a blank `qpc` (SKU57).

## 2. Format problems

Each format problem counted across the full files, and a check that `src/standardize.py` fixes it
without changing what the data means. Standardizing only rewrites values, so row counts must not change.

In [7]:
std = standardize_all(raw)
for name in raw:
    print(f"{name}: {len(raw[name])} rows before, {len(std[name])} after")

parts: 101 rows before, 101 after
inventory: 309 rows before, 309 after
supply: 113 rows before, 113 after
demand: 516 rows before, 516 after


### Why read everything as text (ASSUMPTIONS 2.2)

In [8]:
default_read = pd.read_csv(DATA_DIR / RAW_FILES["inventory"])
print("on_hand_qty read by default as:", default_read["on_hand_qty"].dtype)
on_hand_text = raw["inventory"]["on_hand_qty"]
print("values with a comma:", list(on_hand_text[on_hand_text.str.contains(",")]))
month_cells = raw["demand"][FORECAST_MONTHS]
print("forecast cells that say N/A:", (month_cells == "N/A").sum().sum())
print("forecast cells that are blank:", (month_cells == "").sum().sum())

on_hand_qty read by default as: str
values with a comma: ['1,041', '6,072']
forecast cells that say N/A: 12
forecast cells that are blank: 30


**What we see:** a default read quietly makes `on_hand_qty` a text column because of two commas,
and it would merge the `N/A` and blank cells into one "missing". So we read text and convert it ourselves.

### Part numbers (ASSUMPTIONS 3.1)

In [9]:
for name, table in raw.items():
    part_text = table["part_number"]
    odd = part_text[~part_text.str.fullmatch(r"SKU[1-9]\d*")]
    print(f"{name}: {odd.nunique()} odd spellings, e.g. {sorted(odd.unique())[:5]}")

all_part_text = pd.concat([table["part_number"].str.strip() for table in raw.values()])
zero_padded = sorted(all_part_text[all_part_text.str.match(r"(?i)SKU-?0")].unique())
print(len(zero_padded), "zero-padded spellings:", zero_padded)

parts: 4 odd spellings, e.g. [' SKU93', 'SKU001', 'SKU0049', 'SKU0053']
inventory: 15 odd spellings, e.g. [' SKU24', ' SKU49', ' SKU99', 'SKU-25', 'SKU-42']
supply: 5 odd spellings, e.g. ['SKU-53', 'SKU0022', 'sku29', 'sku61', 'sku92']
demand: 25 odd spellings, e.g. [' SKU24', ' SKU29', ' SKU3', ' SKU33', ' SKU55']
14 zero-padded spellings: ['SKU001', 'SKU0022', 'SKU0034', 'SKU0049', 'SKU005', 'SKU0052', 'SKU0053', 'SKU006', 'SKU0071', 'SKU0076', 'SKU008', 'SKU0080', 'SKU0083', 'SKU0086']


In [10]:
master_parts = set(std["parts"]["part_number"])
for name in raw:
    parts_here = std[name]["part_number"]
    print(f"{name}: {raw[name]['part_number'].nunique()} spellings -> {parts_here.nunique()} parts, "
          f"not in parts.csv: {sorted(set(parts_here) - master_parts)}")

for name, key in GRAIN.items():
    print(f"{name}: {raw[name].duplicated().sum()} exact copies in raw, "
          f"{std[name].duplicated(key).sum()} repeated keys after standardizing")

parts: 100 spellings -> 100 parts, not in parts.csv: []
inventory: 117 spellings -> 102 parts, not in parts.csv: ['SKU901', 'SKU902']
supply: 67 spellings -> 66 parts, not in parts.csv: ['SKU903']
demand: 119 spellings -> 94 parts, not in parts.csv: []
parts: 1 exact copies in raw, 1 repeated keys after standardizing
inventory: 5 exact copies in raw, 5 repeated keys after standardizing
supply: 4 exact copies in raw, 4 repeated keys after standardizing
demand: 2 exact copies in raw, 2 repeated keys after standardizing


**What we see:** after standardizing, every part matches `parts.csv` except SKU901, SKU902 and SKU903.
The only repeated keys are rows that were already exact copies, so dropping leading zeros merged nothing that shouldn't be merged.

### Locations (ASSUMPTIONS 3.2)

In [11]:
location_columns = [("inventory", "location"), ("supply", "destination"), ("demand", "location"), ("demand", "ship_to")]
frames = []
for name, column in location_columns:
    print(f"{name}.{column}: {raw[name][column].nunique()} raw spellings")
    frames.append(pd.DataFrame({"raw": raw[name][column].map(repr), "standard": std[name][column]}))
locations = pd.concat(frames)
pd.crosstab(locations["raw"], locations["standard"])

inventory.location: 16 raw spellings
supply.destination: 11 raw spellings
demand.location: 16 raw spellings
demand.ship_to: 12 raw spellings


standard,,GDC,PDC1,PDC2,PDC3
raw,,,,,
'',304,0,0,0,0
'GDC ',0,14,0,0,0
'GDC',0,372,0,0,0
'Global DC',0,14,0,0,0
'PDC 1',0,0,8,0,0
'PDC 2',0,0,0,13,0
'PDC 3',0,0,0,0,9
'PDC-1',0,0,9,0,0
'PDC-2',0,0,0,8,0


**What we see:** each raw spelling lands on exactly one of the four sites, with nothing left over.
The empty `''` column is `ship_to` on Direct rows, which is blank by design.

### Demand type (ASSUMPTIONS 3.3)

In [12]:
pd.crosstab(raw["demand"]["demand_type"].map(repr), std["demand"]["demand_type"])

demand_type,Direct,Replenishment
demand_type,,
'DIRECT',11,0
'Direct ',10,0
'Direct',277,0
'REPLENISHMENT',0,7
'Replen',0,6
'Replenishment',0,198
'direct',6,0
'replenishment ',0,1


In [13]:
demand = std["demand"]
replen = demand[demand["demand_type"] == "Replenishment"]
direct = demand[demand["demand_type"] == "Direct"]
print(f"Replenishment rows: {len(replen)}, at the GDC: {(replen['location'] == 'GDC').sum()}, "
      f"with a PDC in ship_to: {replen['ship_to'].isin(PDCS).sum()}")
print(f"Direct rows: {len(direct)}, with a ship_to: {(direct['ship_to'] != '').sum()}")
spelled_replen = raw["demand"]["demand_type"] == "Replen"
demand.loc[spelled_replen, ["part_number", "location", "demand_type", "ship_to"]]

Replenishment rows: 212, at the GDC: 212, with a PDC in ship_to: 212
Direct rows: 304, with a ship_to: 0


,part_number,location,demand_type,ship_to
108,SKU9,GDC,Replenishment,PDC2
117,SKU84,GDC,Replenishment,PDC2
118,SKU97,GDC,Replenishment,PDC2
356,SKU45,GDC,Replenishment,PDC2
363,SKU19,GDC,Replenishment,PDC3
499,SKU31,GDC,Replenishment,PDC1


**What we see:** 8 spellings become 2. Every Replenishment row is at the GDC with a PDC in `ship_to`,
and no Direct row has a `ship_to`, which is exactly the task's definition. The 6 `Replen` rows fit the same pattern.

### Planner and part type (ASSUMPTIONS 3.4)

In [14]:
pd.crosstab(raw["parts"]["planner"].map(repr), std["parts"]["planner"])

planner,Alex Rivera,Daniel Okafor,Marcus Chen,Priya Patel,Sofia Novak
planner,,,,,
'Alex Rivera',20,0,0,0,0
'DANIEL OKAFOR',0,2,0,0,0
'Daniel Okafor',0,23,0,0,0
'Marcus Chen ',0,0,1,0,0
'Marcus Chen',0,0,15,0,0
'Priya Patel',0,0,0,15,0
'SOFIA NOVAK',0,0,0,0,1
'Sofia Novak',0,0,0,0,22
'alex rivera',1,0,0,0,0


In [15]:
pd.crosstab(raw["parts"]["part_type"].map(repr), std["parts"]["part_type"])

part_type,Body Panel,Brake,Electrical,Fastener,Glass,HVAC,Lighting,Seat,Suspension,Wheel & Tire
part_type,,,,,,,,,,
'Body Panel',10,0,0,0,0,0,0,0,0,0
'Brake',0,10,0,0,0,0,0,0,0,0
'Electrical',0,0,10,0,0,0,0,0,0,0
'Fastener',0,0,0,10,0,0,0,0,0,0
'Glass',0,0,0,0,9,0,0,0,0,0
'HVAC',0,0,0,0,0,10,0,0,0,0
'Lighting',0,0,0,0,0,0,8,0,0,0
'Seat',0,0,0,0,0,0,0,10,0,0
'Suspension',0,0,0,0,0,0,0,0,10,0


**What we see:** 10 planner spellings become 5 planners, and 13 part types become 10.
The variants differ only in case and spaces.

### Quantity text (ASSUMPTIONS 3.5)

In [16]:
quantity_columns = {"parts": ["qpc"], "inventory": ["on_hand_qty"], "supply": ["qty"], "demand": FORECAST_MONTHS}
rows = []
for name, columns in quantity_columns.items():
    text = raw[name][columns].stack()
    numbers = std[name][columns].stack()
    rows.append({
        "file": name,
        "cells": len(text),
        "with_comma": text.str.contains(",").sum(),
        "ending_in_.0": text.str.endswith(".0").sum(),
        "blank_or_N/A": text.isin(["", "N/A"]).sum(),
        "real_fractions_after_parsing": (numbers.dropna() % 1 != 0).sum(),
    })
pd.DataFrame(rows)

,file,cells,with_comma,ending_in_.0,blank_or_N/A,real_fractions_after_parsing
0,parts,101,0,0,2,0
1,inventory,309,2,33,0,0
2,supply,113,5,20,0,0
3,demand,12384,73,1440,42,0


**What we see:** commas and `.0` endings are formatting only, since no quantity has a real fraction.
Blank and `N/A` cells stay missing for now. What they mean is a cleaning decision (section 3).

### Dates (ASSUMPTIONS 3.6)

In [17]:
eta_text = raw["supply"]["eta_date"].str.strip()
layout = pd.Series("YYYY-MM-DD", index=eta_text.index)
layout.loc[eta_text.str.contains("/")] = "MM/DD/YYYY"
layout.loc[eta_text.str.contains("[A-Za-z]")] = "DD-Mon-YYYY"
layout.value_counts()

YYYY-MM-DD     76
DD-Mon-YYYY    19
MM/DD/YYYY     18
Name: count, dtype: int64

In [18]:
slash_dates = eta_text[eta_text.str.contains("/")]
first_number = slash_dates.str.split("/").str[0].astype(int)
second_number = slash_dates.str.split("/").str[1].astype(int)
# a slash date is ambiguous when both numbers could be a month and they differ
ambiguous = slash_dates[(first_number <= 12) & (second_number <= 12) & (first_number != second_number)]

other_etas = std["supply"]["eta_date"].drop(ambiguous.index)
print("All other ETAs fall between", other_etas.min().date(), "and", other_etas.max().date())
pd.DataFrame({
    "raw": ambiguous,
    "read_as_month_day": pd.to_datetime(ambiguous, format="%m/%d/%Y").dt.date,
    "read_as_day_month": pd.to_datetime(ambiguous, format="%d/%m/%Y").dt.date,
})

All other ETAs fall between 2026-10-25 and 2027-01-12


,raw,read_as_month_day,read_as_day_month
6,12/02/2026,2026-12-02,2026-02-12
18,01/10/2027,2027-01-10,2027-10-01
28,12/05/2026,2026-12-05,2026-05-12
32,01/10/2027,2027-01-10,2027-10-01
52,11/06/2026,2026-11-06,2026-06-11
64,11/02/2026,2026-11-02,2026-02-11
104,11/05/2026,2026-11-05,2026-05-11


**What we see:** 7 slash dates could be read two ways. As month/day they all fall inside the range of the other ETAs.
As day/month, five would land months before the 2026-10-22 snapshot and two in Oct 2027. Month/day is the only reading that fits.

## 3. Cleaning questions

Each block shows one problem and the evidence around it. The rule chosen for each one is recorded
in `ASSUMPTIONS.md` (section 4) and applied in `src/clean.py`.

### Missing values per file

In [19]:
for name, table in raw.items():
    other_columns = [column for column in table.columns if column not in FORECAST_MONTHS]
    missing = table[other_columns].isin(["", "N/A"]).sum()
    print(f"{name}: {missing[missing > 0].to_dict()}")

month_missing = raw["demand"][FORECAST_MONTHS].isin(["", "N/A"])
print("demand month cells:", month_missing.sum().sum())
print("missing cells per month:", month_missing.sum().to_dict())

parts: {'qpc': 2}
inventory: {}
supply: {}
demand: {'ship_to': 304}
demand month cells: 42
missing cells per month: {'2026-11': 0, '2026-12': 0, '2027-01': 0, '2027-02': 2, '2027-03': 2, '2027-04': 3, '2027-05': 2, '2027-06': 3, '2027-07': 3, '2027-08': 1, '2027-09': 0, '2027-10': 1, '2027-11': 2, '2027-12': 1, '2028-01': 4, '2028-02': 3, '2028-03': 3, '2028-04': 3, '2028-05': 1, '2028-06': 2, '2028-07': 3, '2028-08': 1, '2028-09': 0, '2028-10': 2}


**What we see:** inventory and supply have no missing cells. `ship_to` is blank only on Direct rows, by design.
`qpc` is blank for 2 parts, and 42 forecast cells are missing, none of them in the first three months.

### Duplicate rows

In [20]:
for name in raw:
    print(f"{name}: {raw[name].duplicated().sum()} extra copies")
raw["supply"][raw["supply"].duplicated(keep=False)].sort_values(["po_number", "po_line"])

parts: 1 extra copies
inventory: 5 extra copies
supply: 4 extra copies
demand: 2 extra copies


,po_number,po_line,part_number,destination,source,qty,eta_date
73,PO1006,10,SKU43,GDC,Supplier,1136,04-Jan-2027
112,PO1006,10,SKU43,GDC,Supplier,1136,04-Jan-2027
4,PO1011,10,SKU43,PDC3,GDC Transfer,4432,17-Nov-2026
38,PO1011,10,SKU43,PDC3,GDC Transfer,4432,17-Nov-2026
7,PO1012,20,SKU50,GDC,Supplier,"1,103",2027-01-04
67,PO1012,20,SKU50,GDC,Supplier,"1,103",2027-01-04
47,PO1044,10,SKU58,PDC 2,Supplier,58,2026-11-05
91,PO1044,10,SKU58,PDC 2,Supplier,58,2026-11-05


**What we see:** every duplicate is an exact copy in every column. The largest is a 4,432-piece GDC Transfer
(PO1011 line 10) listed twice, so keeping both copies would double it.

### Part/location pairs with no inventory row

In [21]:
inventory = std["inventory"]
inventory_pairs = set(zip(inventory["part_number"], inventory["location"]))
demand_pairs = set(zip(demand["part_number"], demand["location"]))
supply_pairs = set(zip(std["supply"]["part_number"], std["supply"]["destination"]))
print("pairs in inventory:", len(inventory_pairs))
print("pairs with demand but no inventory row:", len(demand_pairs - inventory_pairs))
print("pairs with only supply (no inventory, no demand):", sorted(supply_pairs - inventory_pairs - demand_pairs))
print("all pairs across the three files:", len(inventory_pairs | demand_pairs | supply_pairs))

pairs in inventory: 304
pairs with demand but no inventory row: 41
pairs with only supply (no inventory, no demand): [('SKU903', 'PDC1')]
all pairs across the three files: 346


**What we see:** 41 part/location pairs have demand but no inventory row, and one pair has only incoming supply
(SKU903 at PDC1). The task asks for every SKU at every location, so these belong in the output too.

### Negative on-hand

In [22]:
negative = inventory.loc[inventory["on_hand_qty"] < 0, ["part_number", "location", "on_hand_qty"]].copy()
demand_next_3_months = demand[FORECAST_MONTHS[:3]].sum(axis=1).groupby([demand["part_number"], demand["location"]]).sum()
incoming = std["supply"].groupby(["part_number", "destination"])["qty"].sum()
negative_pairs = list(zip(negative["part_number"], negative["location"]))
negative["demand_next_3_months"] = [demand_next_3_months.get(pair, 0) for pair in negative_pairs]
negative["incoming_supply"] = [incoming.get(pair, 0) for pair in negative_pairs]
negative

,part_number,location,on_hand_qty,demand_next_3_months,incoming_supply
65,SKU46,GDC,-11.0,673.0,0.0
137,SKU92,PDC2,-18.0,84.0,49.0
160,SKU75,PDC1,-15.0,2727.0,0.0
208,SKU52,PDC1,-4.0,437.0,0.0


**What we see:** 4 rows, from -4 to -18 pieces. All four have demand in the next three months and three have
nothing incoming, so they are stockouts under any rule. The rule only shifts the later projection by up to 18 pieces.

### Units in each file (ASSUMPTIONS 1)

The task says every quantity is in pieces. These checks confirm that and show where pallets come in.
A quantity is a whole number of pallets when it divides evenly by `qpc`. By luck that happens about 1 time in `qpc`
(1 in 4 for a part with 4 pieces per pallet), so each stream is compared with that rate.

In [23]:
part_master = std["parts"].drop_duplicates("part_number")
qpc_by_part = part_master.set_index("part_number")["qpc"]

# exact copies (see Duplicate rows) would be counted twice, so drop them for this check
on_hand = inventory.drop_duplicates()
supply = std["supply"].drop_duplicates()
month_cells = demand.drop_duplicates().melt(id_vars=["part_number", "demand_type"], value_vars=FORECAST_MONTHS,
                                            value_name="qty")
streams = {
    "on-hand": on_hand.rename(columns={"on_hand_qty": "qty"}),
    "supply from a supplier": supply[supply["source"] == "Supplier"],
    "supply in transit from the GDC": supply[supply["source"] == "GDC Transfer"],
    "Direct demand": month_cells[month_cells["demand_type"] == "Direct"],
    "Replenishment (planned GDC shipments)": month_cells[month_cells["demand_type"] == "Replenishment"],
}
rows = []
for name, table in streams.items():
    qpc = table["part_number"].map(qpc_by_part)
    # zero divides by anything and negatives are a cleaning question, so test positive values only
    tested = (table["qty"] > 0) & (qpc > 0)
    whole = table.loc[tested, "qty"] % qpc[tested] == 0
    rows.append({
        "stream": name,
        "positive_values": tested.sum(),
        "whole_pallets": whole.sum(),
        "whole_pallets_%": round(100 * whole.mean()),
        "expected_by_luck_%": round(100 * (1 / qpc[tested]).mean()),
    })
pd.DataFrame(rows)

,stream,positive_values,whole_pallets,whole_pallets_%,expected_by_luck_%
0,on-hand,279,38,14,15
1,supply from a supplier,64,7,11,12
2,supply in transit from the GDC,42,5,12,14
3,Direct demand,6585,878,13,15
4,Replenishment (planned GDC shipments),2633,2633,100,17


**What we see:** only the Replenishment rows stand out. All 2,633 positive planned shipments are whole pallets, against 17% by luck.
On-hand, supplier POs, in-transit GDC Transfers and Direct demand all sit near the luck rate, so they are loose pieces
(PO1051, for example, sends 37 pieces of SKU6, which has 50 per pallet). The whole-pallet plan is a pattern in this data,
not a rule from the task, so nothing gets rounded because of it.

Next, the scale. A file in pallets would look tiny next to the others for parts with big pallets, so we compare across pallet sizes.

In [24]:
# if a file counted pallets, parts with big pallets would look far smaller than the rest
size_bands = [0, 5, 20, 100, 1000]
forecast = demand.drop_duplicates()
# rough weekly demand for this check only: next 3 months over 13 weeks, negatives counted as 0
forecast = forecast.assign(total_24m=forecast[FORECAST_MONTHS].clip(lower=0).sum(axis=1),
                           weekly_next_3m=forecast[FORECAST_MONTHS[:3]].clip(lower=0).sum(axis=1) / 13)
direct_by_pair = forecast[forecast["demand_type"] == "Direct"].set_index(["part_number", "location"])
planned = forecast[forecast["demand_type"] == "Replenishment"].set_index(["part_number", "ship_to"])["total_24m"]
planned.index.names = ["part_number", "location"]

plan_vs_sales = pd.concat({"planned": planned, "pdc_direct": direct_by_pair["total_24m"]}, axis=1).dropna()
plan_vs_sales = plan_vs_sales[plan_vs_sales["pdc_direct"] > 0]
plan_vs_sales["ratio"] = plan_vs_sales["planned"] / plan_vs_sales["pdc_direct"]
plan_band = pd.cut(plan_vs_sales.index.get_level_values("part_number").map(qpc_by_part), size_bands)

cover = pd.concat({"on_hand": on_hand.set_index(["part_number", "location"])["on_hand_qty"],
                   "weekly_direct": direct_by_pair["weekly_next_3m"]}, axis=1).dropna()
cover = cover[(cover["on_hand"] > 0) & (cover["weekly_direct"] > 0)]
cover["weeks"] = cover["on_hand"] / cover["weekly_direct"]
cover_band = pd.cut(cover.index.get_level_values("part_number").map(qpc_by_part), size_bands)

by_band = pd.DataFrame({
    "planned_pairs": plan_vs_sales.groupby(plan_band, observed=True).size(),
    "planned_vs_pdc_direct": plan_vs_sales.groupby(plan_band, observed=True)["ratio"].median().round(2),
    "on_hand_pairs": cover.groupby(cover_band, observed=True).size(),
    "on_hand_weeks": cover.groupby(cover_band, observed=True)["weeks"].median().round(1),
})
by_band.index.name = "pieces_per_pallet"
by_band

,planned_pairs,planned_vs_pdc_direct,on_hand_pairs,on_hand_weeks
pieces_per_pallet,,,,
"(0, 5]",58,0.92,65,9.5
"(5, 20]",84,0.95,103,7.6
"(20, 100]",36,0.92,37,7.5
"(100, 1000]",25,1.00,28,8.7


**What we see:** in every pallet-size band, the plan sends a PDC roughly what it sells over 24 months (median 0.92 to 1.00),
and on-hand covers a similar 7.5 to 9.5 weeks of Direct demand. If either file counted pallets, parts with more than 100 pieces
per pallet would look over 100 times smaller. So every file counts pieces. Inventory, demand and supply calculations use pieces,
WOS is reported in weeks, and `qpc` is used only to derive pallet equivalents for inventory and capacity reporting.

The whole-pallet table only tested positive months. Looking at every month, which Replenishment rows have one
that is not a whole number of pallets?

In [25]:
replen_qpc = replen["part_number"].map(qpc_by_part)
has_qpc = replen_qpc > 0
leftover = replen.loc[has_qpc, FORECAST_MONTHS].mod(replen_qpc[has_qpc], axis=0).fillna(0)
not_whole = (leftover != 0).any(axis=1)
print("replenishment rows with a usable qpc:", has_qpc.sum())
print("rows with a month that is not a whole number of pallets:", not_whole.sum())
replen.loc[not_whole[not_whole].index, ["part_number", "location", "ship_to"]]

replenishment rows with a usable qpc: 203
rows with a month that is not a whole number of pallets: 2


,part_number,location,ship_to
122,SKU2,GDC,PDC3
211,SKU98,GDC,PDC2


**What we see:** two rows, SKU2 to PDC3 and SKU98 to PDC2. Those are exactly the rows with a negative month (next block).
Every other planned month is a whole number of pallets, which also helps with the `qpc` question below.

### Negative forecast cells

In [26]:
def flagged_cells(flags: pd.DataFrame) -> pd.DataFrame:
    """List the forecast cells marked True in flags, with the months on either side."""
    last_index = len(FORECAST_MONTHS) - 1
    rows = []
    for month_index, month in enumerate(FORECAST_MONTHS):
        for row_index in flags.index[flags[month]]:
            before = demand.at[row_index, FORECAST_MONTHS[month_index - 1]] if month_index > 0 else None
            after = demand.at[row_index, FORECAST_MONTHS[month_index + 1]] if month_index < last_index else None
            rows.append({"part_number": demand.at[row_index, "part_number"], "location": demand.at[row_index, "location"],
                         "demand_type": demand.at[row_index, "demand_type"], "ship_to": demand.at[row_index, "ship_to"],
                         "month": month, "value": demand.at[row_index, month],
                         "month_before": before, "month_after": after})
    return pd.DataFrame(rows)


negative_cells = flagged_cells(demand[FORECAST_MONTHS] < 0)
negative_cells["qpc"] = negative_cells["part_number"].map(qpc_by_part)
negative_cells

,part_number,location,demand_type,ship_to,month,value,month_before,month_after,qpc
0,SKU1,GDC,Direct,,2027-03,-5.0,5.0,16.0,10.0
1,SKU2,GDC,Replenishment,PDC3,2027-05,-6.0,500.0,0.0,250.0
2,SKU60,PDC3,Direct,,2027-06,-2.0,4.0,5.0,8.0
3,SKU98,GDC,Replenishment,PDC2,2027-07,-12.0,704.0,824.0,8.0
4,SKU37,PDC3,Direct,,2028-04,-7.0,2.0,1.0,2.0
5,SKU30,PDC2,Direct,,2028-05,-5.0,4.0,5.0,4.0


**What we see:** 6 cells, from -2 to -12, none in the first three months. The 4 Direct ones sit between small positive
months of the same size (SKU30: 4, -5, 5), so they look like sign errors. The 2 Replenishment ones don't look like real
shipments either. A shipment can't be negative, and they break the whole-pallet pattern that every other planned month
follows (SKU2 moves in 250-piece pallets, and its month is -6).

### Missing forecast cells

In [27]:
missing_flags = demand[FORECAST_MONTHS].isna()
missing_cells = flagged_cells(missing_flags)
print("by demand type:", missing_cells["demand_type"].value_counts().to_dict())
print("in the first 3 months:", missing_cells["month"].isin(FORECAST_MONTHS[:3]).sum())
side_by_side = missing_flags & missing_flags.shift(1, axis=1, fill_value=False)
print("rows with two missing months side by side:", side_by_side.any(axis=1).sum())
missing_cells

by demand type: {'Direct': 27, 'Replenishment': 15}
in the first 3 months: 0


rows with two missing months side by side:

 0


,part_number,location,demand_type,ship_to,month,value,month_before,month_after
0,SKU99,GDC,Replenishment,PDC2,2027-02,NaN,0.0,0.0
1,SKU28,GDC,Direct,,2027-02,NaN,5.0,3.0
2,SKU52,GDC,Replenishment,PDC2,2027-03,NaN,0.0,0.0
3,SKU62,GDC,Direct,,2027-03,NaN,10.0,4.0
4,SKU5,GDC,Replenishment,PDC2,2027-04,NaN,0.0,4.0
5,SKU75,GDC,Replenishment,PDC1,2027-04,NaN,0.0,1250.0
6,SKU84,PDC1,Direct,,2027-04,NaN,25.0,39.0
7,SKU5,PDC3,Direct,,2027-05,NaN,1.0,1.0
8,SKU80,GDC,Direct,,2027-05,NaN,2.0,2.0
9,SKU84,PDC3,Direct,,2027-06,NaN,2.0,9.0


**What we see:** 27 Direct and 15 Replenishment cells. None in the first three months, and never two in a row.
Most sit between normal months (SKU44 at the GDC: 561, missing, 452), so reading them as 0 would understate demand.

### Missing or zero `qpc`

In [28]:
bad_qpc = part_master[part_master["qpc"].isna() | (part_master["qpc"] == 0)]
rows = []
for _, part in bad_qpc.iterrows():
    shipments = replen.loc[replen["part_number"] == part["part_number"], FORECAST_MONTHS].stack()
    shipment_sizes = sorted(shipments[shipments > 0].astype(int).unique())
    same_type = part_master[(part_master["part_type"] == part["part_type"]) & (part_master["qpc"] > 0)]
    rows.append({
        "part_number": part["part_number"],
        "part_type": part["part_type"],
        "qpc": part["qpc"],
        "on_hand_pieces": inventory.loc[inventory["part_number"] == part["part_number"], "on_hand_qty"].sum(),
        "replenishment_shipment_sizes": shipment_sizes,
        "largest_size_dividing_all": math.gcd(*shipment_sizes),
        "qpc_of_same_part_type": sorted(same_type["qpc"].astype(int)),
    })
pd.DataFrame(rows)

,part_number,part_type,qpc,on_hand_pieces,replenishment_shipment_sizes,largest_size_dividing_all,qpc_of_same_part_type
0,SKU57,HVAC,NaN,90.0,"[6, 12]",6,"[6, 6, 6, 6, 6, 8, 8, 8]"
1,SKU34,Glass,NaN,96.0,"[6, 12, 18, 24]",6,"[4, 4, 4, 4, 8, 10, 10, 10, 10]"
2,SKU5,HVAC,0.0,55.0,"[4, 8, 12]",4,"[6, 6, 6, 6, 6, 8, 8, 8]"


**What we see:** every other planned shipment is a whole number of pallets, so the pallet size most likely divides these
parts' planned shipments too. The largest size that fits is 6 for SKU57 and SKU34 and 4 for SKU5. A smaller size such as 2
would also divide them, so this is a best guess, not proof. The usual size for the part type (8 for Glass, 6 for HVAC) would
not fit SKU34's or SKU5's shipments. Together these parts hold 241 pieces, about 45 pallets, so the choice matters little.

### Parts not in `parts.csv`

In [29]:
parts_in_use = set(inventory["part_number"]) | set(std["supply"]["part_number"])
unknown_parts = sorted(parts_in_use - set(part_master["part_number"]))
display(inventory[inventory["part_number"].isin(unknown_parts)])
display(std["supply"][std["supply"]["part_number"].isin(unknown_parts)])
print("forecast rows for these parts:", demand["part_number"].isin(unknown_parts).sum())
po_numbers = sorted(std["supply"]["po_number"].unique())
print("PO numbers run from", po_numbers[0], "to", po_numbers[-2], "plus", po_numbers[-1])

,part_number,location,on_hand_qty,snapshot_date
101,SKU901,PDC2,35.0,2026-10-22
156,SKU902,GDC,120.0,2026-10-22


,po_number,po_line,part_number,destination,source,qty,eta_date
106,PO9999,10,SKU903,PDC1,Supplier,60.0,2026-11-20


forecast rows for these parts: 0
PO numbers run from PO1001 to PO1054 plus PO9999


**What we see:** SKU901 (35 pieces at PDC2) and SKU902 (120 at the GDC) are on hand, and SKU903 has 60 pieces
arriving at PDC1 on PO9999, the only PO number outside the PO10xx range. None of them has a forecast.